# Domain G: GLM-Based Encoding Analysis

This notebook addresses three questions using pre-fitted ridge-regularized GLM encoding models stored in the zarr multimodal data:
- **G1:** Do cell types differ in the relative balance of visual vs. state-driven activity?
- **G2:** What do condition-specific GLM kernels reveal about temporal encoding across cell types?
- **G3:** Can GLM residuals (y − y_hat) reveal unmodeled computation?

**Data:** Zarr multimodal stores containing GLM coefficients, scores, y (observed), y_hat (predicted), y_hat_visual, and y_hat_state per cell per session.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, spearmanr, pearsonr
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

sns.set_context('talk')
sns.set_style('whitegrid')

# ── Constants ──
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}

# ── Load GLM scores and cell-type labels from all mice and sessions ──
glm_records = []
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    n_total_cells = len(subclass_names)
    
    for sess in SESSIONS:
        glm = z[f'ophys/drifting_gratings/{sess}/glm']
        score_total = glm['score/score_total'][:]
        score_visual = glm['score/score_visual'][:]
        score_state = glm['score/score_state'][:]
        score_test = glm['score/score_test'][:]
        n_glm = len(score_total)
        
        # GLM may be fit on a subset of cells
        # Use index mapping if available; otherwise assume first n_glm cells
        for ci in range(n_glm):
            sc_idx = ci if ci < n_total_cells else 0
            glm_records.append({
                'mouse_id': mouse_id, 'session': sess,
                'cell_idx': ci,
                'subclass': subclass_names[sc_idx] if sc_idx < n_total_cells else 'unknown',
                'score_total': score_total[ci],
                'score_visual': score_visual[ci],
                'score_state': score_state[ci],
                'score_test': score_test[ci],
                'visual_fraction': score_visual[ci] / max(score_total[ci], 1e-6),
                'state_fraction': score_state[ci] / max(score_total[ci], 1e-6),
            })

glm_df = pd.DataFrame(glm_records)
glm_df['subclass_short'] = glm_df['subclass'].map(SUBCLASS_SHORT)
present_subclasses = [s for s in SUBCLASS_ORDER if s in glm_df['subclass'].unique()]

print(f"Total GLM records: {len(glm_df)} ({glm_df['mouse_id'].nunique()} mice × "
      f"{glm_df['session'].nunique()} sessions)")
print(f"\nGLM score_total summary per session:")
print(glm_df.groupby(['mouse_id', 'session'])['score_total'].describe()[['count', 'mean', 'max']].round(4))

## G1: Visual vs. State-Driven Activity by Cell Type

Decompose each cell's encoding into visual and state (running/pupil) components using the pre-fitted GLM R² scores. Test whether cell types differ in their visual/state balance.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# G1  Visual vs State R² Decomposition by Cell Type
# ══════════════════════════════════════════════════════════════════════

# Session 1 data for primary analysis
s1_df = glm_df[glm_df['session'] == 'session_1'].copy()
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Scatter: visual R² vs state R², colored by subclass
ax = axes[0, 0]
for sc in present_subclasses:
    mask = s1_df['subclass'] == sc
    ax.scatter(s1_df.loc[mask, 'score_visual'], s1_df.loc[mask, 'score_state'],
               c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc], alpha=0.6, s=20, edgecolors='none')
ax.plot([0, 0.5], [0, 0.5], 'k--', alpha=0.3)
ax.set_xlabel('Visual R²')
ax.set_ylabel('State R²')
ax.set_title('G1: Visual vs State Encoding (Session 1)', fontweight='bold')
ax.legend(fontsize=7, loc='upper left')

# 2. Violin: visual fraction by subclass
ax = axes[0, 1]
sns.violinplot(data=s1_df[s1_df['subclass_short'].isin(short_order)],
               x='subclass_short', y='visual_fraction', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.set_title('G1: Visual Fraction of Total R²', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Visual R² / Total R²')
ax.tick_params(axis='x', rotation=45)

# 3. Violin: total R² by subclass
ax = axes[1, 0]
sns.violinplot(data=s1_df[s1_df['subclass_short'].isin(short_order)],
               x='subclass_short', y='score_total', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.set_title('G1: Total GLM R² by Subclass', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Total R²')
ax.tick_params(axis='x', rotation=45)

# 4. Cross-session stability: visual R² session 1 vs session 3
ax = axes[1, 1]
s1_vis = glm_df[glm_df['session'] == 'session_1'].set_index(['mouse_id', 'cell_idx'])['score_visual']
s3_vis = glm_df[glm_df['session'] == 'session_3'].set_index(['mouse_id', 'cell_idx'])['score_visual']
common_idx = s1_vis.index.intersection(s3_vis.index)
if len(common_idx) > 10:
    ax.scatter(s1_vis.loc[common_idx], s3_vis.loc[common_idx], alpha=0.3, s=10, c='steelblue')
    r, p = pearsonr(s1_vis.loc[common_idx], s3_vis.loc[common_idx])
    ax.plot([0, s1_vis.max()], [0, s1_vis.max()], 'k--', alpha=0.4)
    ax.set_xlabel('Visual R² (Session 1)')
    ax.set_ylabel('Visual R² (Session 3)')
    ax.set_title(f'G1: Cross-Session Stability (r={r:.3f}, p={p:.2e})', fontweight='bold')

plt.tight_layout()
plt.show()

# ── Statistics ──
print("Visual R² by subclass (Kruskal-Wallis):")
groups = [s1_df.loc[s1_df['subclass'] == s, 'score_visual'].dropna().values
          for s in present_subclasses if (s1_df['subclass'] == s).sum() >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"  H={stat:.2f}, p={p:.2e}")

print("\nState R² by subclass (Kruskal-Wallis):")
groups = [s1_df.loc[s1_df['subclass'] == s, 'score_state'].dropna().values
          for s in present_subclasses if (s1_df['subclass'] == s).sum() >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"  H={stat:.2f}, p={p:.2e}")

print("\nMean scores by subclass (Session 1):")
print(s1_df.groupby('subclass_short')[['score_visual', 'score_state', 'score_total']].mean().round(4))